In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      cross_val_score)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (BaggingClassifier, AdaBoostClassifier,
                               GradientBoostingClassifier,
                               StackingClassifier)
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, roc_curve, auc)

# --- 1. Load dataset -------------------------------------------------------
wdbc_bundle    = load_breast_cancer()
feature_matrix = pd.DataFrame(wdbc_bundle.data,
                               columns=wdbc_bundle.feature_names)
diagnosis_vec  = pd.Series(wdbc_bundle.target, name='diagnosis')

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    feature_matrix, diagnosis_vec,
    test_size=0.20, random_state=42, stratify=diagnosis_vec
)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# --- 2. Bagging sweep -------------------------------------------------------
best_bag_acc, best_bag_cfg, bag_log = 0.0, {}, []
for n in [25, 50, 100, 200]:
    for ms in [0.7, 0.9, 1.0]:
        for mf in [0.7, 1.0]:
            clf = BaggingClassifier(
                estimator=DecisionTreeClassifier(random_state=42),
                n_estimators=n, max_samples=ms, max_features=mf,
                random_state=42, n_jobs=-1)
            acc = cross_val_score(
                clf, X_train_raw, y_train_raw,
                cv=skf, scoring='accuracy').mean()
            f1 = cross_val_score(
                clf, X_train_raw, y_train_raw,
                cv=skf, scoring='f1').mean()
            bag_log.append({'n_estimators': n, 'max_samples': ms,
                            'max_features': mf,
                            'avg_cv_acc': round(acc*100,2),
                            'avg_cv_f1':  round(f1,4)})
            if acc > best_bag_acc:
                best_bag_acc = acc
                best_bag_cfg = {'n_estimators': n, 'max_samples': ms,
                                'max_features': mf}

print(f"Best Bagging: {best_bag_cfg}, CV={round(best_bag_acc*100,2)}%")

# --- 3. AdaBoost sweep ------------------------------------------------------
best_ada_acc, best_ada_cfg, ada_log = 0.0, {}, []
for n in [50, 100, 200]:
    for lr in [0.1, 0.5, 1.0]:
        clf = AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1,random_state=42),
            n_estimators=n, learning_rate=lr, random_state=42)
        acc = cross_val_score(
            clf, X_train_raw, y_train_raw,
            cv=skf, scoring='accuracy').mean()
        f1 = cross_val_score(
            clf, X_train_raw, y_train_raw,
            cv=skf, scoring='f1').mean()
        ada_log.append({'n_estimators': n, 'learning_rate': lr,
                        'avg_cv_acc': round(acc*100,2),
                        'avg_cv_f1':  round(f1,4)})
        if acc > best_ada_acc:
            best_ada_acc = acc
            best_ada_cfg = {'n_estimators': n, 'learning_rate': lr}

print(f"Best AdaBoost: {best_ada_cfg}, CV={round(best_ada_acc*100,2)}%")

# --- 4. Gradient Boosting sweep --------------------------------------------
best_gb_acc, best_gb_cfg, gb_log = 0.0, {}, []
for n in [50, 100, 150]:
    for lr in [0.05, 0.1, 0.2]:
        for d in [2, 3]:
            clf = GradientBoostingClassifier(
                n_estimators=n, learning_rate=lr,
                max_depth=d, random_state=42)
            acc = cross_val_score(
                clf, X_train_raw, y_train_raw,
                cv=skf, scoring='accuracy').mean()
            f1 = cross_val_score(
                clf, X_train_raw, y_train_raw,
                cv=skf, scoring='f1').mean()
            gb_log.append({'n_estimators': n, 'learning_rate': lr,
                           'max_depth': d,
                           'avg_cv_acc': round(acc*100,2),
                           'avg_cv_f1':  round(f1,4)})
            if acc > best_gb_acc:
                best_gb_acc = acc
                best_gb_cfg = {'n_estimators': n, 'learning_rate': lr,
                               'max_depth': d}

print(f"Best GradBoost: {best_gb_cfg}, CV={round(best_gb_acc*100,2)}%")

# --- 5. Stacking sweep -----------------------------------------------------
meta_lr = LogisticRegression(max_iter=1000, random_state=42)
stacking_combos = [
    ('SVM + DT', [
        ('svm', make_pipeline(StandardScaler(),
                SVC(probability=True, random_state=42))),
        ('dt',  DecisionTreeClassifier(max_depth=5, random_state=42))]),
    ('SVM + NB + DT', [
        ('svm', make_pipeline(StandardScaler(),
                SVC(probability=True, random_state=42))),
        ('nb',  GaussianNB()),
        ('dt',  DecisionTreeClassifier(max_depth=5, random_state=42))]),
    ('SVM + NB', [
        ('svm', make_pipeline(StandardScaler(),
                SVC(probability=True, random_state=42))),
        ('nb',  GaussianNB())]),
    ('NB + DT', [
        ('nb',  GaussianNB()),
        ('dt',  DecisionTreeClassifier(max_depth=5, random_state=42))]),
]

best_stk_acc, best_stk_cfg, stk_log = 0.0, {}, []
for combo_name, estimators in stacking_combos:
    clf = StackingClassifier(estimators=estimators,
                              final_estimator=meta_lr,
                              cv=5, n_jobs=-1)
    acc = cross_val_score(
        clf, X_train_raw, y_train_raw,
        cv=skf, scoring='accuracy').mean()
    f1 = cross_val_score(
        clf, X_train_raw, y_train_raw,
        cv=skf, scoring='f1').mean()
    stk_log.append({'base_models': combo_name,
                    'meta_learner': 'Logistic Regression',
                    'avg_cv_acc': round(acc*100,2),
                    'avg_cv_f1':  round(f1,4)})
    if acc > best_stk_acc:
        best_stk_acc = acc
        best_stk_cfg = {'name': combo_name,
                        'estimators': estimators}

print(f"Best Stacking: {best_stk_cfg['name']}, CV={round(best_stk_acc*100,2)}%")

# --- 6. Train champion models on full training split ----------------------
champ_bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    **best_bag_cfg, random_state=42, n_jobs=-1)
champ_ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
    **best_ada_cfg, random_state=42)
champ_gb  = GradientBoostingClassifier(**best_gb_cfg, random_state=42)
champ_stk = StackingClassifier(
    estimators=best_stk_cfg['estimators'],
    final_estimator=meta_lr, cv=5, n_jobs=-1)

for model in [champ_bag, champ_ada, champ_gb, champ_stk]:
    model.fit(X_train_raw, y_train_raw)

# --- 7. Test-set evaluation -----------------------------------------------
def report_metrics(y_true, y_pred, y_proba, tag):
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    print(f"\n{tag}")
    print(f"  Accuracy  : {round(accuracy_score(y_true,y_pred)*100,2)}%")
    print(f"  Precision : {round(precision_score(y_true,y_pred)*100,2)}%")
    print(f"  Recall    : {round(recall_score(y_true,y_pred)*100,2)}%")
    print(f"  F1-Score  : {round(f1_score(y_true,y_pred)*100,2)}%")
    print(f"  AUC-ROC   : {round(auc(fpr,tpr),4)}")
    print(f"  Conf Matrix:\n{confusion_matrix(y_true,y_pred)}")

report_metrics(y_test_raw, champ_bag.predict(X_test_raw),
               champ_bag.predict_proba(X_test_raw)[:,1], 'Bagging')
report_metrics(y_test_raw, champ_ada.predict(X_test_raw),
               champ_ada.predict_proba(X_test_raw)[:,1], 'AdaBoost')
report_metrics(y_test_raw, champ_gb.predict(X_test_raw),
               champ_gb.predict_proba(X_test_raw)[:,1], 'Gradient Boosting')
report_metrics(y_test_raw, champ_stk.predict(X_test_raw),
               champ_stk.predict_proba(X_test_raw)[:,1], 'Stacking')